Basics
------
This covers a fundamental use-case for DN<sup>3</sup>. Here, the TIDNet architecture from *Kostas & Rudzicz 2020
(https://doi.org/10.1088/1741-2552/abb7a7)* is evaluated using the Physionet motor dataset in a
*"leave multiple subjects out"* (aka person-stratified cross validation) training procedure.
This means, the number of subjects are divided into approximately evenly numbered sets, and the test performance of
each set is evaluated. The remaining sets for each test will be used to develop a neural network classifier.

Here we will focus on classifying imagined hand vs. foot movement, which are the runs 6, 10 and 14. So create a
configuration for the configuratron that specifies a single dataset, the length of the trials to cut for the dataset
 (6s), the events we are looking for (T1 and T2), and then to exclude all those sessions that are not 6, 10 and 14
(the sessions by default are listed as "S{subject number}R{run number}.edf". Finally, due to some issues with a few
people's recordings in the dataset, we can ignore those troublemakers. The following is the contents of a file named
"my_config.yml".

```yaml
Configuratron:
  preload: True

use_gpu: False

mmidb:
  name: "Physionet MMIDB"
  toplevel: /path/to/eegmmidb
  tmin: 0
  tlen: 6
  data_max: 0.001
  data_min: -0.001
  events:
    - T1
    - T2
  exclude_sessions:
    - "*R0[!48].edf"  # equivalently "*R0[1235679].edf"
    - "*R1[!2].edf"   # equivalently "*R1[134].edf"
  exclude_people:
    - S088
    - S090
    - S092
    - S100
  train_params:
    epochs: 7
    batch_size: 4
  lr: 0.0001
  folds: 5
```

Notice that in addition to the dataset and special `Configuratron` section we also include a `train_params` with the
dataset.
This last part is definitely *not mandatory*, it is an arbitrarily named additional set of configuration values that
will be used in our experiment, put there for convenience. See it in action below, but if you don't like it,
don't use it.

In [1]:
from dn3.configuratron import ExperimentConfig
from dn3.trainable.processes import StandardClassification
from dn3.trainable.models import TIDNet

# Since we are doing a lot of loading, this is nice to suppress some tedious information
import mne
mne.set_log_level(False)

/home/hice1/salammurillo3/.conda/envs/dn3_legacy/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


First things first, we use `ExperimentConfig`, and the subsequently constructed `DatasetConfig` to rapidly construct
our `dataset`.

In [2]:
config_filename = 'my_config.yml'
experiment = ExperimentConfig(config_filename)
ds_config = experiment.datasets['mmidb']

dataset = ds_config.auto_construct_dataset()

Adding additional configuration entries: dict_keys(['train_params', 'lr'])
Configuratron found 1 datasets.


Scanning /home/hice1/salammurillo3/scratch/bendr/physionet. If there are a lot of files, this may take a while...: 100%|██████████| 4/4 [00:00<00:00, 18.69it/s, extension=.gdf]


Creating dataset of 315 Preloaded Epoched recordings from 105 people.


Loading Physionet MMIDB: 100%|██████████| 105/105 [00:18<00:00,  5.64person/s]


>> Physionet MMIDB | DSID: None | 105 people | 4408 trials | 90 channels | 960 samples/trial | 160.0Hz | 0 transforms
Constructed 1 channel maps
Used by 315 recordings:
EEG (original(new)): Fc5.(FC5) Fc3.(FC3) Fc1.(FC1) Fcz.(FCZ) Fc2.(FC2) Fc4.(FC4) Fc6.(FC6) C5..(C5) C3..(C3) C1..(C1) Cz..(CZ) C2..(C2) C4..(C4) C6..(C6) Cp5.(CP5) Cp3.(CP3) Cp1.(CP1) Cpz.(CPZ) Cp2.(CP2) Cp4.(CP4) Cp6.(CP6) Fp1.(FP1) Fpz.(FPZ) Fp2.(FP2) Af7.(AF7) Af3.(AF3) Afz.(AFZ) Af4.(AF4) Af8.(AF8) F7..(F7) F5..(F5) F3..(F3) F1..(F1) Fz..(FZ) F2..(F2) F4..(F4) F6..(F6) F8..(F8) Ft7.(FT7) Ft8.(FT8) T7..(T7) T8..(T8) T9..(T9) T10.(T10) Tp7.(TP7) Tp8.(TP8) P7..(P7) P5..(P5) P3..(P3) P1..(P1) Pz..(PZ) P2..(P2) P4..(P4) P6..(P6) P8..(P8) Po7.(PO7) Po3.(PO3) Poz.(POZ) Po4.(PO4) Po8.(PO8) O1..(O1) Oz..(OZ) O2..(O2) Iz..(IZ) 
EOG (original(new)): 
REF (original(new)): 
EXTRA (original(new)): 
Heuristically Assigned: Fc5.(FC5)  Fc3.(FC3)  Fc1.(FC1)  Fcz.(FCZ)  Fc2.(FC2)  Fc4.(FC4)  Fc6.(FC6)  C5..(C5)  C3..(C3)  C1..(C1)  Cz

Let's create a function that makes a new model for each set of training people and a trainable process for
`StandardClassification`. The `cuda` argument handles whether the GPU is used to train the neural network if a
cuda-compatible PyTorch installation is operational.

In [3]:
def make_model_and_process():
    tidnet = TIDNet.from_dataset(dataset)
    return StandardClassification(tidnet, cuda=experiment.use_gpu, learning_rate=ds_config.lr)

That's pretty much it! We use a helper that initializes a TIDNet from any `Dataset/Thinker/EpochedRecording` and wrap
it with a `StandardClassification` process. Invoking this process will train the classifier.
Have a look through all the `trainable.processes`, they can wrap the *same model* to learn in stages (e.g. some sort
 of fine-tuning procedure from a general model -- covered in `examples/finetuning.ipynb`).

Now, we loop through five folds, *leaving multiple subjects out* (`dataset.lmso()`), fit the classifier,
then check the test results.

In [4]:
results = list()
for training, validation, test in dataset.lmso(ds_config.train_params.folds):
    process = make_model_and_process()

    process.fit(training_dataset=training, validation_dataset=validation, **ds_config.train_params)

    results.append(process.evaluate(test)['Accuracy'])

print(results)
print("Average accuracy: {:.2%}".format(sum(results)/len(results)))

Training:   >> Physionet MMIDB | DSID: None | 63 people | 2646 trials | 90 channels | 960 samples/trial | 160.0Hz | 0 transforms
Validation: >> Physionet MMIDB | DSID: None | 21 people | 880 trials | 90 channels | 960 samples/trial | 160.0Hz | 0 transforms
Test:       >> Physionet MMIDB | DSID: None | 21 people | 882 trials | 90 channels | 960 samples/trial | 160.0Hz | 0 transforms
Creating TIDNet using: 90 channels x 960 samples at 160.0Hz | 2 targets


/home/hice1/salammurillo3/.conda/envs/dn3_legacy/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading data with 2 additional workers


Iteration: 662batches [00:04, 135.49batches/s, Accuracy=0.634, loss=0.691, lr=3.53e-5, momentum=0.917]                    
Epoch:  14%|█▍        | 1/7 [00:04<?, ?epoch/s]

Training: End of Epoch 1 | Accuracy: 63.39% | loss: 0.691 | lr: 3.539e-05 | momentum: 0.917 | epoch: 1.000 |


...

Training: End of Epoch 7 | Accuracy: 90.73% | loss: 0.216 | lr: 2.565e-06 | momentum: 0.947 | epoch: 7.000 |


Predicting: 100%|██████████| 221/221 [00:00<00:00, 241.18it/s]
Epoch: 8epoch [00:40,  5.73s/epoch]                    


Validation: End of Epoch 7 | Accuracy: 85.94% | loss: 0.362 |
Loading best model...


Predicting: 100%|██████████| 880/880 [00:02<00:00, 394.15it/s]

[0.9092970490455627, 0.8854875564575195, 0.9081632494926453, 0.8480725884437561, 0.8488636612892151]
Average accuracy: 88.00%


Check out how we passed the train_params to `.fit()`, we could specify more arguments for `.fit()` by just adding them
to this section in the configuration.

Say we wanted to *instead*, get the performance of *each* person in the test fold independently. We could replace the
code above with a very simple alternative, that *leaves one subject out* or `.loso()`, that specifically.
It would look like:

In [5]:
results = dict()

for training, validation, test in dataset.lmso(ds_config.train_params.folds):
    process = make_model_and_process()

    process.fit(
        training_dataset=training,
        validation_dataset=validation,
        epochs=ds_config.train_params.epochs,
        batch_size=ds_config.train_params.batch_size
    )

    for _, _, test_thinker in test.loso():
        metrics = process.evaluate(test_thinker)
        results[test_thinker.person_id] = metrics['Accuracy']

print(results)
print("Average accuracy: {:.2%}".format(sum(results.values()) / len(results)))

Training:   >> Physionet MMIDB | DSID: None | 63 people | 2646 trials | 90 channels | 960 samples/trial | 160.0Hz | 0 transforms
Validation: >> Physionet MMIDB | DSID: None | 21 people | 880 trials | 90 channels | 960 samples/trial | 160.0Hz | 0 transforms
Test:       >> Physionet MMIDB | DSID: None | 21 people | 882 trials | 90 channels | 960 samples/trial | 160.0Hz | 0 transforms


/home/hice1/salammurillo3/.conda/envs/dn3_legacy/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Creating TIDNet using: 90 channels x 960 samples at 160.0Hz | 2 targets
Loading data with 2 additional workers


Iteration: 662batches [00:04, 146.46batches/s, Accuracy=0.65, loss=0.683, lr=3.53e-5, momentum=0.917]                    
Epoch:  14%|█▍        | 1/7 [00:04<?, ?epoch/s]

Training: End of Epoch 1 | Accuracy: 65.05% | loss: 0.683 | lr: 3.539e-05 | momentum: 0.917 | epoch: 1.000 |


...

Training:   >> Physionet MMIDB | DSID: None | 19 people | 796 trials | 90 channels | 960 samples/trial | 160.0Hz | 0 transforms
Validation: Person S107 - 42 trials | 0 transforms
Test:       Person S108 - 42 trials | 0 transforms


Predicting: 100%|██████████| 42/42 [00:00<00:00, 386.34it/s]


Training:   >> Physionet MMIDB | DSID: None | 19 people | 796 trials | 90 channels | 960 samples/trial | 160.0Hz | 0 transforms
Validation: Person S108 - 42 trials | 0 transforms
Test:       Person S109 - 42 trials | 0 transforms


Predicting: 100%|██████████| 42/42 [00:00<00:00, 378.87it/s]

{'S001': 0.9523809552192688, 'S002': 0.9523809552192688, 'S003': 0.976190447807312, 'S004': 0.9523809552192688, 'S005': 0.9285714030265808, 'S006': 1.0, 'S007': 1.0, 'S008': 0.8333333134651184, 'S009': 0.9285714030265808, 'S010': 0.6666666865348816, 'S011': 0.9285714030265808, 'S012': 0.9285714030265808, 'S013': 0.7857142686843872, 'S014': 0.9523809552192688, 'S015': 0.9523809552192688, 'S016': 0.738095223903656, 'S017': 0.9523809552192688, 'S018': 0.976190447807312, 'S019': 0.8809523582458496, 'S020': 0.976190447807312, 'S021': 0.6428571343421936, 'S022': 1.0, 'S023': 0.761904776096344, 'S024': 0.8809523582458496, 'S025': 0.5, 'S026': 0.6904761791229248, 'S027': 0.8809523582458496, 'S028': 0.976190447807312, 'S029': 0.976190447807312, 'S030': 0.976190447807312, 'S031': 0.9285714030265808, 'S032': 1.0, 'S033': 1.0, 'S034': 0.9047619104385376, 'S035': 0.9047619104385376, 'S036': 0.9047619104385376, 'S037': 0.9285714030265808, 'S038': 0.4761904776096344, 'S039': 0.8571428656578064, 'S040

Try filling in your own values to `my_config.yml` to run these examples on your next read through.